Exercise 1

In [4]:
!pip install pandas numpy scikit-learn gensim tabulate nltk

   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ------- -------------------------------- 4.5/24.4 MB 23.3 MB/s eta 0:00:01
   ------------- -------------------------- 8.4/24.4 MB 20.6 MB/s eta 0:00:01
   ------------------- -------------------- 12.1/24.4 MB 19.9 MB/s eta 0:00:01
   ----------------------------- ---------- 17.8/24.4 MB 21.9 MB/s eta 0:00:01
   -------------------------------------- - 23.6/24.4 MB 23.3 MB/s eta 0:00:01
   ---------------------------------------- 24.4/24.4 MB 22.5 MB/s eta 0:00:00

   ---------------------------------------- 0/2 [smart_open]
   ---------------------------------------- 0/2 [smart_open]
   ---------------------------------------- 0/2 [smart_open]
   -------------------- ------------------- 1/2 [gensim]
   -------------------- ------------------- 1/2 [gensim]
   -------------------- ------------------- 1/2 [gensim]
   -------------------- ------------------- 1/2 [gensim]
   -------------------- ------------------- 1/

In [6]:
import re
import string
import numpy as np
import pandas as pd
import nltk

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics import silhouette_score
from gensim.models import Word2Vec
from tabulate import tabulate
from collections import Counter

In [7]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Arif\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt.zip.


True

In [8]:
def preprocess_text(text):
    text = str(text).lower()   # lowercase
    text = re.sub(r"http\S+|www\S+", "", text)   # remove URLs
    text = re.sub(r"\d+", "", text)   # remove numbers
    text = text.translate(str.maketrans("", "", string.punctuation))   # remove punctuation
    tokens = text.split()
    tokens = [word for word in tokens if word not in ENGLISH_STOP_WORDS]   # remove stopwords
    return " ".join(tokens)

In [9]:
dataset = [
    "I love playing football on the weekends",
    "I enjoy hiking and camping in the mountains",
    "I like to read books and watch movies",
    "I prefer playing video games over sports",
    "I love listening to music and going to concerts"
]

In [10]:
dataset_clean = [preprocess_text(doc) for doc in dataset]

print("Original text:")
for doc in dataset:
    print(doc)

print("\nPreprocessed text:")
for doc in dataset_clean:
    print(doc)

Original text:
I love playing football on the weekends
I enjoy hiking and camping in the mountains
I like to read books and watch movies
I prefer playing video games over sports
I love listening to music and going to concerts

Preprocessed text:
love playing football weekends
enjoy hiking camping mountains
like read books watch movies
prefer playing video games sports
love listening music going concerts


In [11]:
vectorizer_raw = TfidfVectorizer()
X_raw = vectorizer_raw.fit_transform(dataset)

k = 2
km_raw = KMeans(n_clusters=k, random_state=42, n_init=10)
labels_raw = km_raw.fit_predict(X_raw)

print("TF-IDF without preprocessing:")
for doc, label in zip(dataset, labels_raw):
    print(f"Cluster {label}: {doc}")

TF-IDF without preprocessing:
Cluster 0: I love playing football on the weekends
Cluster 0: I enjoy hiking and camping in the mountains
Cluster 1: I like to read books and watch movies
Cluster 0: I prefer playing video games over sports
Cluster 1: I love listening to music and going to concerts


In [12]:
vectorizer_clean = TfidfVectorizer()
X_clean = vectorizer_clean.fit_transform(dataset_clean)

km_clean = KMeans(n_clusters=k, random_state=42, n_init=10)
labels_clean = km_clean.fit_predict(X_clean)

print("\nTF-IDF with preprocessing:")
for doc, label in zip(dataset, labels_clean):
    print(f"Cluster {label}: {doc}")


TF-IDF with preprocessing:
Cluster 1: I love playing football on the weekends
Cluster 0: I enjoy hiking and camping in the mountains
Cluster 0: I like to read books and watch movies
Cluster 0: I prefer playing video games over sports
Cluster 1: I love listening to music and going to concerts


In [13]:
print("\nTop terms per TF-IDF cluster:")
order_centroids = km_clean.cluster_centers_.argsort()[:, ::-1]
terms = vectorizer_clean.get_feature_names_out()

for i in range(k):
    print(f"\nCluster {i}:")
    for ind in order_centroids[i, :10]:
        print(terms[ind])


Top terms per TF-IDF cluster:

Cluster 0:
mountains
hiking
enjoy
camping
video
games
prefer
sports
watch
books

Cluster 1:
love
football
weekends
music
listening
going
concerts
playing
watch
read


In [14]:
def lab_purity(labels):
    total_samples = len(labels)
    cluster_label_counts = [Counter(labels)]
    purity = sum(max(cluster.values()) for cluster in cluster_label_counts) / total_samples
    return purity

In [15]:
print("\nTF-IDF lab-style purity without preprocessing:", lab_purity(labels_raw))
print("TF-IDF lab-style purity with preprocessing:", lab_purity(labels_clean))


TF-IDF lab-style purity without preprocessing: 0.6
TF-IDF lab-style purity with preprocessing: 0.6


In [16]:
if len(set(labels_raw)) > 1:
    print("TF-IDF silhouette without preprocessing:", silhouette_score(X_raw, labels_raw))

if len(set(labels_clean)) > 1:
    print("TF-IDF silhouette with preprocessing:", silhouette_score(X_clean, labels_clean))

TF-IDF silhouette without preprocessing: 0.060065242166785625
TF-IDF silhouette with preprocessing: 0.020603126928711683


In [17]:
tokenized_raw = [doc.split() for doc in dataset]

w2v_raw_model = Word2Vec(
    sentences=tokenized_raw,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

def get_doc_embedding(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

X_w2v_raw = np.array([get_doc_embedding(doc.split(), w2v_raw_model) for doc in dataset])

km_w2v_raw = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_w2v_raw = km_w2v_raw.fit_predict(X_w2v_raw)

print("\nWord2Vec without preprocessing:")
for doc, label in zip(dataset, labels_w2v_raw):
    print(f"Cluster {label}: {doc}")


Word2Vec without preprocessing:
Cluster 1: I love playing football on the weekends
Cluster 1: I enjoy hiking and camping in the mountains
Cluster 0: I like to read books and watch movies
Cluster 1: I prefer playing video games over sports
Cluster 0: I love listening to music and going to concerts


C:\Users\Arif\Music\New folder\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [18]:
tokenized_clean = [doc.split() for doc in dataset_clean]

w2v_clean_model = Word2Vec(
    sentences=tokenized_clean,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

X_w2v_clean = np.array([get_doc_embedding(doc.split(), w2v_clean_model) for doc in dataset_clean])

km_w2v_clean = KMeans(n_clusters=2, random_state=42, n_init=10)
labels_w2v_clean = km_w2v_clean.fit_predict(X_w2v_clean)

print("\nWord2Vec with preprocessing:")
for doc, label in zip(dataset, labels_w2v_clean):
    print(f"Cluster {label}: {doc}")


Word2Vec with preprocessing:
Cluster 0: I love playing football on the weekends
Cluster 0: I enjoy hiking and camping in the mountains
Cluster 0: I like to read books and watch movies
Cluster 0: I prefer playing video games over sports
Cluster 1: I love listening to music and going to concerts


C:\Users\Arif\Music\New folder\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [19]:
print("\nWord2Vec lab-style purity without preprocessing:", lab_purity(labels_w2v_raw))
print("Word2Vec lab-style purity with preprocessing:", lab_purity(labels_w2v_clean))

if len(set(labels_w2v_raw)) > 1:
    print("Word2Vec silhouette without preprocessing:", silhouette_score(X_w2v_raw, labels_w2v_raw))

if len(set(labels_w2v_clean)) > 1:
    print("Word2Vec silhouette with preprocessing:", silhouette_score(X_w2v_clean, labels_w2v_clean))


Word2Vec lab-style purity without preprocessing: 0.6
Word2Vec lab-style purity with preprocessing: 0.8
Word2Vec silhouette without preprocessing: 0.1260512
Word2Vec silhouette with preprocessing: 0.06577299


Exercise 2

In [31]:
df = pd.read_csv("customer_complaints_1 (1).csv")
print(df.head())
print(df.columns)

                        author      posted_on  rating  \
0  Alantae of Chesterfeild, MI  Nov. 22, 2016       1   
1     Vera of Philadelphia, PA  Nov. 19, 2016       1   
2  Sarah of Rancho Cordova, CA  Nov. 17, 2016       1   
3     Dennis of Manchester, NH  Nov. 16, 2016       1   
4         Ryan of Bellevue, WA  Nov. 14, 2016       1   

                                                text  
0  I used to love Comcast. Until all these consta...  
1  I'm so over Comcast! The worst internet provid...  
2  If I could give them a negative star or no sta...  
3  I've had the worst experiences so far since in...  
4  Check your contract when you sign up for Comca...  
Index(['author', 'posted_on', 'rating', 'text'], dtype='object')


In [21]:
complaints = df["text"].dropna().astype(str).tolist()
complaints_clean = [preprocess_text(text) for text in complaints]

print("Total complaints:", len(complaints))
print("\nSample original complaint:")
print(complaints[0])

print("\nSample cleaned complaint:")
print(complaints_clean[0])

Total complaints: 19

Sample original complaint:
I used to love Comcast. Until all these constant updates. My internet and cable crash a lot at night, and sometimes during the day, some channels don't even work and on demand sometimes don't play either. I wish they will do something about it. Because just a few mins ago, the internet have crashed for about 20 mins for no reason. I'm tired of it and thinking about switching to Wow or something. Please do not get Xfinity.

Sample cleaned complaint:
used love comcast constant updates internet cable crash lot night day channels dont work demand dont play wish just mins ago internet crashed mins reason im tired thinking switching wow xfinity


In [22]:
tfidf_vectorizer_csv = TfidfVectorizer(max_features=500)
X_csv_tfidf = tfidf_vectorizer_csv.fit_transform(complaints_clean)

k_csv = 3   # you can test 2, 3, 4, 5
km_csv_tfidf = KMeans(n_clusters=k_csv, random_state=42, n_init=10)
labels_csv_tfidf = km_csv_tfidf.fit_predict(X_csv_tfidf)

df["tfidf_cluster"] = labels_csv_tfidf

In [23]:
print(df[["text", "tfidf_cluster"]].head(10))

                                                text  tfidf_cluster
0  I used to love Comcast. Until all these consta...              0
1  I'm so over Comcast! The worst internet provid...              0
2  If I could give them a negative star or no sta...              0
3  I've had the worst experiences so far since in...              0
4  Check your contract when you sign up for Comca...              1
5  Thank God. I am changing to Dish. They gave me...              1
6  I Have been a long time customer and only have...              1
7  There is a malfunction on the DVR manager whic...              1
8  Charges overwhelming. Comcast service rep was ...              2
9  I have had cable, DISH, and U-verse, etc. in t...              0


In [24]:
print("\nTop terms per complaint TF-IDF cluster:")
order_centroids_csv = km_csv_tfidf.cluster_centers_.argsort()[:, ::-1]
terms_csv = tfidf_vectorizer_csv.get_feature_names_out()

for i in range(k_csv):
    print(f"\nCluster {i}:")
    for ind in order_centroids_csv[i, :10]:
        print(terms_csv[ind])


Top terms per complaint TF-IDF cluster:

Cluster 0:
internet
comcast
cable
service
tech
just
months
im
come
mins

Cluster 1:
contract
mbps
xfinity
years
customer
blast
comcast
speed
signed
told

Cluster 2:
rude
service
rep
day
speed
joke
customer
local
people
helpful


In [25]:
print("\nComplaint TF-IDF lab-style purity:", lab_purity(labels_csv_tfidf))

if len(set(labels_csv_tfidf)) > 1:
    print("Complaint TF-IDF silhouette:", silhouette_score(X_csv_tfidf, labels_csv_tfidf))


Complaint TF-IDF lab-style purity: 0.42105263157894735
Complaint TF-IDF silhouette: 0.01868209022618384


In [26]:
tokenized_complaints = [doc.split() for doc in complaints_clean]

w2v_csv_model = Word2Vec(
    sentences=tokenized_complaints,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4
)

X_csv_w2v = np.array([
    get_doc_embedding(doc.split(), w2v_csv_model) for doc in complaints_clean
])

km_csv_w2v = KMeans(n_clusters=k_csv, random_state=42, n_init=10)
labels_csv_w2v = km_csv_w2v.fit_predict(X_csv_w2v)

df["w2v_cluster"] = labels_csv_w2v

C:\Users\Arif\Music\New folder\Lib\site-packages\sklearn\cluster\_kmeans.py:1419: UserWarning: KMeans is known to have a memory leak on Windows with MKL, when there are less chunks than available threads. You can avoid it by setting the environment variable OMP_NUM_THREADS=1.
  warnings.warn(


In [27]:
print(df[["text", "w2v_cluster"]].head(10))

                                                text  w2v_cluster
0  I used to love Comcast. Until all these consta...            2
1  I'm so over Comcast! The worst internet provid...            2
2  If I could give them a negative star or no sta...            0
3  I've had the worst experiences so far since in...            0
4  Check your contract when you sign up for Comca...            2
5  Thank God. I am changing to Dish. They gave me...            2
6  I Have been a long time customer and only have...            2
7  There is a malfunction on the DVR manager whic...            0
8  Charges overwhelming. Comcast service rep was ...            1
9  I have had cable, DISH, and U-verse, etc. in t...            2


In [28]:
print("\nComplaint Word2Vec lab-style purity:", lab_purity(labels_csv_w2v))

if len(set(labels_csv_w2v)) > 1:
    print("Complaint Word2Vec silhouette:", silhouette_score(X_csv_w2v, labels_csv_w2v))


Complaint Word2Vec lab-style purity: 0.631578947368421
Complaint Word2Vec silhouette: 0.07976512


In [29]:
df.to_csv("customer_complaints_clustered_output.csv", index=False)
print("Saved successfully.")

Saved successfully.
